# Introducing Z3

In this notebook we are going to solve the following exercise using the constraint solver `Z3`.

   * I have as many brothers as I have sisters, but
   * my sister has twice as many brothers as she has sisters.
   
How many children does my father have?

We load the Java API of `Z3` from *Maven Central*, which is the standard repository for Java libraries.  The magic command `%maven` downloads the library when it is executed for the first time.  Then, we import the package `com.microsoft.z3`.

In [ ]:
%maven tools.aqua:z3-turnkey:4.14.1
import com.microsoft.z3.*;

All `Z3` objects are created by a *context*.  We call this context `Z3`.

In [ ]:
Context Z3 = new Context();

In `Z3`, every variable has to be declared within a context. In this case, all our variables are integer valued. The method `Z3.mkIntConst(name)` declares an integer variable with the given name.

The object `Z3.mkIntConst("boys")` is the `Z3` representation of a constraint variable. It has the type `IntExpr`.  We assign this object to the variable `boys`.

In [ ]:
IntExpr boys  = Z3.mkIntConst("boys");
IntExpr girls = Z3.mkIntConst("girls");

Next, we create a *solver* object.  This is a constraint solver capable of solving constraint satisfaction problems.

In [ ]:
Solver S = Z3.mkSolver();

We can add constraints to this solver object via the method `add`.
We will use two variables:
 - `boys` is the number of my father's sons,
 - `girls` is the number of my father's daughters.
 
Note that the number of my brothers is `boys - 1` as I am not my own brother. Similarly, the number of my sister's sisters is `sisters - 1`.

In Java, we cannot overload standard operators like `+` or `==`. Instead, we use methods of the context like `Z3.mkSub()`, `Z3.mkMul()`, and `Z3.mkEq()` to build the constraints.  Numbers are converted into `Z3` objects using the method `Z3.mkInt()`.
So the statement that I have as many brothers as sisters, but my sister has twice as many brothers as she has sisters is formalized as follows:

In [ ]:
S.add(Z3.mkEq(Z3.mkSub(boys, Z3.mkInt(1)), girls));                         // boys - 1 == girls
S.add(Z3.mkEq(Z3.mkMul(Z3.mkSub(girls, Z3.mkInt(1)), Z3.mkInt(2)), boys));  // 2 * (girls - 1) == boys

The method `check` examines whether the given set of constraints is satisfiable.

In general, it can return one of the following results:
- `SATISFIABLE`   is returned if the problem is solvable,
- `UNSATISFIABLE` is returned if the problem is unsatisfiable,
- `UNKNOWN` is returned if the constraint solver is not powerful enough to solve the given problem.

These values are elements of the *enumeration* `Status`.

In [ ]:
S.check()

To extract the solution of the given problem we use the method `getModel`.

In [ ]:
Model model = S.getModel();

In [ ]:
model.toString()

In order to extract the values for `boys` and `girls`, we use the `eval` method of the model, e.g., `model.eval(boys, true)`.  The second argument `true` specifies that variables without a value are assigned a default value.

This returns a `Z3` expression. To use these results as Java numbers, we first convert them to strings using `toString()` and then parse them using `Integer.parseInt()`.

In [ ]:
int b = Integer.parseInt(model.eval(boys,  true).toString());
int g = Integer.parseInt(model.eval(girls, true).toString());

In [ ]:
System.out.println("My father has " + (b + g) + " children: " + b + " boys and " + g + " girls.");
System.out.println("I have " + b + " - 1 = " + (b - 1) + " brothers and " + g + " sisters.");
System.out.println("My sister has " + g + " - 1 = " + (g - 1) + " sisters and " + b + " = 2 * " + (g - 1) + " brothers.");